In [16]:
import numpy as np
from time import perf_counter
from pynq import Overlay, allocate

class Benchmark:
    def __init__(self):
        self.reset()
        
    def __enter__(self):
        self.entered += 1
        self.runtime[0] = perf_counter()
        return self
        
    def __exit__(self, *exc):
        self.runtime[1] = perf_counter()
        self.acc += self.elapsed
        
    def reset(self):
        self.acc = 0.0
        self.entered = 0
        self.runtime = [0.0, 0.0]
    
    @property
    def elapsed(self):
        return self.runtime[1] - self.runtime[0]
    
    @property
    def total_elapsed(self):
        return self.acc
    
    @property
    def average(self):
        return self.total_elapsed / self.entered

class Buffer:
    def __init__(self, data_dim, pred_dim):
        self.data = allocate(shape=data_dim, dtype=np.float32)
        self.pred = allocate(shape=pred_dim, dtype=np.uint32)
        
    def copy(self, data):
        np.copyto(self.data, data)
        
class DMA:
    def __init__(self, ip):
        self.data = ip.axi_dma_0.sendchannel
        self.pred = ip.axi_dma_1.recvchannel
        
    def __call__(self, buffer):
        self.data.transfer(buffer.data)
        self.pred.transfer(buffer.pred)
        
        self.data.wait()
        self.pred.wait()
        
class MNIST_SLP(Overlay):
    def __init__(self, *args, batch_size, **kwargs):
        super().__init__(*args, **kwargs)
        self.batch_size = batch_size
        self.buffer = Buffer([batch_size, 28, 28], [batch_size])
        self.dma = DMA(self)
                
    def run(self, data):
        self.buffer.copy(data)
        self.dma(self.buffer)
        return self.buffer.pred

In [4]:
mnist_images = np.load("MNIST_DATASET_IMAGE.npy", allow_pickle=True)
mnist_labels = np.load("MNIST_DATASET_LABEL.npy", allow_pickle=True)

In [ ]:
mnist_slp = MNIST_SLP("{your_own_bitstream}.bit", batch_size=140)

In [17]:
mnist_chunk = mnist_images[::mnist_slp.batch_size]
benchmark = Benchmark()
preds = list()

for n, data_chunk in enumerate(mnist_chunk, 1):
    print(f"#{n} kernel calls...", end='\r')
    with benchmark:
        pred = mnist_slp.run(data_chunk)
    preds.extend(pred)

In [18]:
num_hit = (preds==mnist_labels).sum()
num_hit / len(mnist_labels)

0.9315714285714286

In [19]:
f"total {benchmark.total_elapsed:g}s, {benchmark.average:g}s per call"

'total 34.2303s, 0.0684605s per call'